## APARTADO 1) COMPILACIÓN DEL CORPUS Y USO DE PROCESAMIENTO LÉXICO

### Análisis Temporal del Corpus Inicial

Realizamos un análisis básico para descubrir si nuestro Corpus Inicial cuenta con un espaciado temporal suficiente para realizar correctamente este trabajo.

In [16]:
import json
from datetime import datetime

# Lista de los archivos que generaste anteriormente
archivos_json = ["ejemplo_subreddit_travel.json", "ejemplo_subreddit_RandomThoughts.json", "ejemplo_subreddit_unpopularopinion.json",
                  "ejemplo_subreddit_jobs.json", "ejemplo_subreddit_books.json", "ejemplo_subreddit_LeagueOfLegends.json"]

for archivo in archivos_json:
    try:
        with open(archivo, 'r', encoding='utf-8') as f:
            data = json.load(f)
        
        print(f"\n{'='*50}")
        print(f"📅 ANÁLISIS TEMPORAL: r/{data['subreddit']}")
        print(f"{'='*50}")

        # Extraer fechas de creación de las submissions (convertidas de UTC a datetime)
        # created_utc viene en los datos originales del volcado [cite: 58]
        fechas = [datetime.fromtimestamp(s['created_utc']) for s in data['submissions']]
        
        if fechas:
            fechas_ordenadas = sorted(fechas)
            primera = fechas_ordenadas[0]
            ultima = fechas_ordenadas[-1]
            rango_dias = (ultima - primera).days

            print(f"🔹 Primera publicación: {primera.strftime('%Y-%m-%d %H:%M')}")
            print(f"🔹 Última publicación:  {ultima.strftime('%Y-%m-%d %H:%M')}")
            print(f"🔹 Amplitud temporal:    {rango_dias} días")
            
            if rango_dias < 15:
                print("\n⚠️ ALERTA: Todos los hilos son del mismo día. Considera saltar registros en la extracción.")
            else:
                print("\n✅ El corpus presenta variedad temporal.")
        else:
            print("❌ No se encontraron fechas en las submissions.")

    except FileNotFoundError:
        print(f"⚠️ No se encontró el archivo: {archivo}")


📅 ANÁLISIS TEMPORAL: r/travel
🔹 Primera publicación: 2025-01-01 19:49
🔹 Última publicación:  2025-04-12 21:35
🔹 Amplitud temporal:    101 días

✅ El corpus presenta variedad temporal.

📅 ANÁLISIS TEMPORAL: r/RandomThoughts
🔹 Primera publicación: 2025-01-01 02:04
🔹 Última publicación:  2025-03-20 02:56
🔹 Amplitud temporal:    78 días

✅ El corpus presenta variedad temporal.

📅 ANÁLISIS TEMPORAL: r/unpopularopinion
🔹 Primera publicación: 2025-01-01 05:20
🔹 Última publicación:  2025-03-15 23:53
🔹 Amplitud temporal:    73 días

✅ El corpus presenta variedad temporal.

📅 ANÁLISIS TEMPORAL: r/jobs
🔹 Primera publicación: 2025-01-01 05:00
🔹 Última publicación:  2025-03-26 11:47
🔹 Amplitud temporal:    84 días

✅ El corpus presenta variedad temporal.

📅 ANÁLISIS TEMPORAL: r/books
🔹 Primera publicación: 2025-01-01 11:43
🔹 Última publicación:  2025-04-09 21:54
🔹 Amplitud temporal:    98 días

✅ El corpus presenta variedad temporal.

📅 ANÁLISIS TEMPORAL: r/LeagueOfLegends
🔹 Primera publicación: 2

### Detección de comentarios inservibles

Tratamos de detectar y descubrir qué cantidad de comentarios del Corpus Inicial no son utilizables porque correspondan con comentarios de bots, o que incluyan únicamente *URLs* o sean directamente demasiado cortos como para contener información relevante.

In [17]:
import json
import re


def analizar_calidad(texto):
    # Patrones para detectar URLs y Emails
    tiene_url = bool(re.search(r'https?://\S+|www\.\S+', texto, re.IGNORECASE))
    tiene_email = bool(re.search(r'\S+@\S+\.\S+', texto, re.IGNORECASE))
    tiene_eliminado = bool(re.search(r'\[removed\]|\[deleted\]', texto, re.IGNORECASE))
    longitud = len(texto.split()) # Contamos palabras
    bot_declarado = bool(re.search(r'I am a bot|beep[- ]?boop|action was performed automatically|contact the moderators|RemindMe!|opt out|stop instructions', texto, re.IGNORECASE))
    spam_bot = bool(re.search(r'free karma|subscribe|follow my|click here|check out my', texto, re.IGNORECASE))
    return longitud, tiene_url, tiene_email, tiene_eliminado, bot_declarado, spam_bot

total_inservibles = 0
total_general = 0

for archivo in archivos_json:
    with open(archivo, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    total_comentarios = 0
    cortos = 0 # Menos de 5 palabras
    solo_links = 0
    con_email = 0
    eliminados = 0
    es_bot = 0
    
    for submission in data['submissions']:
        for comment in submission.get('comments', []):
            total_comentarios += 1
            cuerpo = comment.get('body', '')
            author = comment.get('author','')
            
            n_palabras, has_url, has_email, has_removed, bot_declarado, spam_bot = analizar_calidad(cuerpo)
            
            if n_palabras < 3:
                cortos += 1
            elif has_url and n_palabras < 3: # Muy corto y con URL suele ser solo spam/link
                solo_links += 1
            elif has_email:
                con_email += 1
            elif has_removed:
                eliminados += 1
            elif bot_declarado or spam_bot or bool(re.search(r'(?i)bot$|_bot$|robot', author, re.IGNORECASE)):
                es_bot += 1

    total_inservibles += cortos + solo_links + con_email + eliminados + es_bot
    total_general += total_comentarios
    print(f"\n{'='*50}")
    print(f"🔍 CALIDAD DEL TEXTO: r/{data['subreddit']}")
    print(f"{'='*50}")
    print(f"✅ Total analizados: {total_comentarios}")
    print(f"⚠️ Comentarios muy cortos (< 3 palabras): {cortos} ({cortos/total_comentarios*100:.1f}%)")
    print(f"🔗 Comentarios que son casi solo URLs: {solo_links}")
    print(f"📧 Comentarios con emails: {con_email}")
    print(f"🚫 Comentarios con eliminados: {eliminados}")
    print(f"🤖 Comentarios que parecen bots o spam: {es_bot}")
    
    if cortos / total_comentarios > 0.2:
        print("💡 Sugerencia: El corpus tiene mucho 'ruido' (mensajes cortos). Deberías filtrar en el siguiente paso.")

print(f'\nComentarios totales: {total_general}. Comentarios inservibles: {total_inservibles}. Comentarios útiles: {total_general-total_inservibles}')


🔍 CALIDAD DEL TEXTO: r/travel
✅ Total analizados: 2100
⚠️ Comentarios muy cortos (< 3 palabras): 138 (6.6%)
🔗 Comentarios que son casi solo URLs: 0
📧 Comentarios con emails: 0
🚫 Comentarios con eliminados: 0
🤖 Comentarios que parecen bots o spam: 16

🔍 CALIDAD DEL TEXTO: r/RandomThoughts
✅ Total analizados: 2100
⚠️ Comentarios muy cortos (< 3 palabras): 402 (19.1%)
🔗 Comentarios que son casi solo URLs: 0
📧 Comentarios con emails: 0
🚫 Comentarios con eliminados: 0
🤖 Comentarios que parecen bots o spam: 74

🔍 CALIDAD DEL TEXTO: r/unpopularopinion
✅ Total analizados: 2100
⚠️ Comentarios muy cortos (< 3 palabras): 172 (8.2%)
🔗 Comentarios que son casi solo URLs: 0
📧 Comentarios con emails: 1
🚫 Comentarios con eliminados: 0
🤖 Comentarios que parecen bots o spam: 76

🔍 CALIDAD DEL TEXTO: r/jobs
✅ Total analizados: 2100
⚠️ Comentarios muy cortos (< 3 palabras): 169 (8.0%)
🔗 Comentarios que son casi solo URLs: 0
📧 Comentarios con emails: 0
🚫 Comentarios con eliminados: 0
🤖 Comentarios que par

### Limpieza inicial

Eliminamos todos los comentarios que sean inservibles según lo detectado anteriormente y nos quedamos con una estructura unificada en cuanto a cantidad de hilos con número mínimo de comentarios por subreddit.

In [18]:
import json
import re
import random

random.seed(2) # Para reproducibilidad

MIN_HILOS = 50
MIN_COMENTARIOS = 20

def es_inservible(comentario):
    texto = comentario.get('body','')
    author = comentario.get('author','')
    """Retorna True si el texto debe ser eliminado."""
    if not texto: return True
    
    # Lógica de detección
    n_palabras = len(texto.split())
    tiene_url = bool(re.search(r'https?://\S+|www\.\S+', texto))
    tiene_email = bool(re.search(r'\S+@\S+\.\S+', texto))
    tiene_eliminado = bool(re.search(r'\[removed\]|\[deleted\]', texto))
    bot_declarado = bool(re.search(r'I am a bot|beep[- ]?boop|action was performed automatically|contact the moderators|RemindMe!|opt out|stop instructions', texto, re.IGNORECASE))
    spam_bot = bool(re.search(r'subscribe|follow my|click here|check out my', texto, re.IGNORECASE))
    author_bot = bool(re.search(r'(?i)bot$|_bot$|robot', author, re.IGNORECASE))
    
    # Criterios de descarte
    if n_palabras < 3: return True
    if tiene_url and n_palabras < 3: return True
    if tiene_email or tiene_eliminado or bot_declarado or spam_bot or author_bot: return True
    
    return False
# --- PASO 1: Filtrado inicial ---
data_estructurada = {} # Estructura: { subreddit: [ [hilo1_coments], [hilo2_coments] ] }

for archivo in archivos_json:
    with open(archivo, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    sub_name = data['subreddit']
    hilos_del_sub = []
    
    for submission in data.get('submissions', []):
        # Filtramos comentarios de este hilo
        comentarios_limpios = [c for c in submission.get('comments', []) if not es_inservible(c)]
        # Solo añadimos el hilo si tiene comentarios después de la limpieza
        if len(comentarios_limpios) >= MIN_COMENTARIOS: # Aseguramos que el hilo tenga al menos M comentarios útiles
            hilos_del_sub.append({
                "info_hilo": {k: v for k, v in submission.items() if k != 'comments'}, # Todo menos los comments viejos
                "comentarios_limpios": comentarios_limpios
            })

    if len(hilos_del_sub) >= MIN_HILOS:    
        data_estructurada[sub_name] = hilos_del_sub

# --- PASO 2: Calcular mínimos para uniformidad ---
# 1. El número mínimo de hilos que tiene el subreddit más "pobre"
min_hilos = min(len(hilos) for hilos in data_estructurada.values())

# 2. El número mínimo de comentarios que tiene el hilo más "pobre" de todo el dataset
# (Para asegurar que todos los hilos puedan tener la misma cantidad)
min_comentarios_por_hilo = 9999
for sub_hilos in data_estructurada.values():
    for h in sub_hilos:
        if len(h['comentarios_limpios']) < min_comentarios_por_hilo:
            min_comentarios_por_hilo = len(h['comentarios_limpios'])

print(f"Configuración uniforme final:")
print(f"Hilos por Subreddit: {min_hilos}")
print(f"Comentarios por Hilo: {min_comentarios_por_hilo}")
print(f"Total por Subreddit: {min_hilos * min_comentarios_por_hilo}")
print(f"Total de Comentarios útiles:{min_hilos*min_comentarios_por_hilo*6} ")

# --- PASO 3: Recorte y exportación ---
for sub, lista_hilos in data_estructurada.items():
    hilos_seleccionados = random.sample(lista_hilos, min_hilos)
    
    final_submissions = []
    for i, h in enumerate(hilos_seleccionados):
        # Muestreamos comentarios
        comentarios_finales = random.sample(h['comentarios_limpios'], min_comentarios_por_hilo)
        
        # Reconstruimos la estructura incluyendo la info original
        submission_completa = h['info_hilo'].copy()
        submission_completa["id_hilo_limpio"] = i
        submission_completa["comments"] = comentarios_finales
        
        final_submissions.append(submission_completa)
    
    output = {
        "subreddit": sub,
        "configuracion": f"{min_hilos}hilos_{min_comentarios_por_hilo}com",
        "submissions": final_submissions
    }
    
    with open(f"completo_{sub}.json", "w", encoding="utf-8") as f:
        json.dump(output, f, ensure_ascii=False, indent=4)

Configuración uniforme final:
Hilos por Subreddit: 59
Comentarios por Hilo: 20
Total por Subreddit: 1180
Total de Comentarios útiles:7080 


Vemos que finalmente nos quedamos con una cifra bastante inferior a lo detectado anteriormente, esto ocurre porque una vez eliminados los 1275 comentarios inservibles detectados anteriormente, debemos de quedarnos con una cantidad igual en cada subreddit de hilos con un número mínimo de comentarios. Esto implica que, si de los 70 hilos de un subreddit, hay 20 hilos que se quedan finalmente con menos de 20 comentarios tras eliminar estos comentarios (umbral que hemos seleccionado como límite), la configuración unificada más grande posible sería de 50 hilos x 20 comentarios por hilo, y debe de hacerse entonces un cribado aleatorio de hilos y comentarios para alcanzar esta configuración en el resto de subreddits. 

En este caso la configuración ha quedado como 59 hilos x 20 comentarios por hilo, contando con un total de 7080 comentarios finalmente.

A este corpus lo llamaremos **Corpus Completo**.

### Limpieza específica

Eliminamos las stopwords, los emojis y cualquier otra palabra que se pueda considerar "basura" del Corpus Completo para quedarnos con una versión "limpia" del mismo y poder utilizarla para comparar si existe una mejora real en las actividades realizadas posteriormente al utilizar la versión limpia.

A este corpus lo llamaremos **Corpus Limpio**.

In [20]:
import json
import re
import emoji
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

nltk.download('punkt')
nltk.download('stopwords')

def limpiar_texto(texto, minusculas=False):
    if not texto:
        return ""
    
    # Eliminamos Emojis
    texto = emoji.replace_emoji(texto, replace='')
    
    # Pasar el texto a minúsculas (opcional)
    if minusculas:
        texto = re.sub(r'[^a-zA-Z\s]', '', texto.lower())

    # Tokenización y Stopwords
    tokens = word_tokenize(texto)
    stop_words = set(stopwords.words('english'))
    
    # Filtrado: quitamos stopwords y palabras de 1 sola letra (basura común)
    palabras_limpias = [w for w in tokens if w not in stop_words and len(w) > 1]
    
    return " ".join(palabras_limpias)

def limpieza_completa(archivos, minusculas=False):
    for archivo in archivos:
        with open(archivo, 'r', encoding='utf-8') as f:
            datos = json.load(f)
        
        for sub in datos.get('submissions',[]):
            if 'title' in sub:
                sub['title'] = limpiar_texto(sub['title'], minusculas=minusculas)
            if 'selftext' in sub:
                sub['selftext'] = limpiar_texto(sub['selftext'], minusculas=minusculas)
            
            for comment in sub.get('comments',[]):
                if 'body' in comment:
                    comment['body'] = limpiar_texto(comment['body'], minusculas=minusculas)
        
        nombre = archivo.replace("completo", "limpio")
        with open(nombre, "w", encoding="utf-8") as f:
            json.dump(datos, f, ensure_ascii=False, indent=4)
    
        print(f"✅ Archivo procesado y guardado como: {nombre}")

mis_archivos = [
    "completo_travel.json", 
    "completo_RandomThoughts.json",
    "completo_unpopularopinion.json",
    "completo_jobs.json",
    "completo_books.json",
    "completo_LeagueOfLegends.json"
]

limpieza_completa(mis_archivos)

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\sergi\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\sergi\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


✅ Archivo procesado y guardado como: limpio_travel.json
✅ Archivo procesado y guardado como: limpio_RandomThoughts.json
✅ Archivo procesado y guardado como: limpio_unpopularopinion.json
✅ Archivo procesado y guardado como: limpio_jobs.json
✅ Archivo procesado y guardado como: limpio_books.json
✅ Archivo procesado y guardado como: limpio_LeagueOfLegends.json
